# CWA Mechanistic Sub-Experiments: Small-Weight Init and Constant-Shift Ablation on CIFAR-10

This notebook demonstrates two mechanistic sub-experiments on 10-layer unnormalized MLPs (no BatchNorm/Dropout) trained on CIFAR-10 using the **Curie-Weiss Activation (CWA)**: a novel activation function with a learned coupling parameter `J` and a closed-form IFT backward pass.

**Sub-Exp A (Small-Weight Init)**: Tests whether reduced init magnitudes allow the mean-field parameter `J·s̄` to reach near-critical values (>0.7).

**Sub-Exp B (Shift Ablation)**: Isolates whether CWA's accuracy comes from inter-neuron coupling (full fixed-point) or just the mean-shift in pre-activations.

Precomputed results from a 9-minute RTX 5090 run are loaded from the mini dataset. The notebook also defines the CWA model and runs a quick smoke test to verify the mechanism.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — not pre-installed on Colab
_pip('loguru')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')
    _pip('torch==2.9.0', '--index-url', 'https://download.pytorch.org/whl/cpu')

In [ ]:
import sys
import math
import json
import time
import gc
import resource
import multiprocessing as mp
from pathlib import Path
from collections import defaultdict

from loguru import logger

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
Path("logs").mkdir(exist_ok=True)
logger.add("logs/run.log", rotation="30 MB", level="DEBUG")

import numpy as np
from scipy import stats
import torch
import torch.nn as nn

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-2f6f35-curie-weiss-activation-formal-verificati/main/round-3/experiment-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
examples = data["datasets"][0]["examples"]
print(f"Loaded {len(examples)} examples from dataset: {data['datasets'][0]['dataset']}")
print(f"Title: {data['metadata']['title']}")

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
# Minimal values for smoke-test / quick demo. Original full-run values commented.

SMOKE_HIDDEN   = 32      # hidden dim for smoke-test MLP  (original: 256)
SMOKE_DEPTH    = 3       # depth for smoke-test MLP        (original: 10)
SMOKE_K_MAX    = 10      # CWA fixed-point iterations      (original: 50)
SMOKE_BATCH    = 4       # batch size for smoke-test       (original: 256)
SEEDS          = [42, 123, 777]   # seeds used in full experiment (unchanged)

## Hardware Detection & Limits

Detects available RAM and GPU, sets a conservative RLIMIT_AS memory ceiling, and verifies CUDA works (RTX 5090 / sm_120 requires PyTorch ≥ 2.7). Falls back to CPU if CUDA is not functional.

In [ ]:
def _container_ram_gb() -> float:
    for p in ["/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"]:
        try:
            v = Path(p).read_text().strip()
            if v != "max" and int(v) < 1_000_000_000_000:
                return int(v) / 1e9
        except (FileNotFoundError, ValueError):
            pass
    import psutil
    return psutil.virtual_memory().total / 1e9

TOTAL_RAM_GB = _container_ram_gb()
HAS_GPU = torch.cuda.is_available()
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if HAS_GPU else 0.0
DEVICE = torch.device("cuda" if HAS_GPU else "cpu")

logger.info(f"RAM: {TOTAL_RAM_GB:.1f} GB | GPU: {HAS_GPU} | VRAM: {VRAM_GB:.1f} GB | device={DEVICE}")

# Set conservative RAM limit
try:
    RAM_BUDGET = int(min(40, TOTAL_RAM_GB * 0.45) * 1024**3)
    resource.setrlimit(resource.RLIMIT_AS, (RAM_BUDGET * 3, RAM_BUDGET * 3))
except (ValueError, resource.error) as e:
    logger.warning(f"Could not set RLIMIT_AS: {e}")

# Verify CUDA actually works (RTX 5090 / sm_120 requires PyTorch >= 2.7)
_CUDA_OK = False
if HAS_GPU:
    try:
        _test = torch.randn(4, 4, device="cuda")
        _ = (_test @ _test)
        _CUDA_OK = True
        logger.info("CUDA verified working")
        _free, _total = torch.cuda.mem_get_info(0)
        VRAM_BUDGET = min(_free * 0.80, _total * 0.85)
        torch.cuda.set_per_process_memory_fraction(min(VRAM_BUDGET / _total, 0.85))
        logger.info(f"VRAM budget: {VRAM_BUDGET/1e9:.1f} GB / {_total/1e9:.1f} GB")
    except Exception as e:
        logger.warning(f"CUDA not functional (sm_120 incompatibility?): {e} — falling back to CPU")
        _CUDA_OK = False

DEVICE = torch.device("cuda" if _CUDA_OK else "cpu")
logger.info(f"Effective device: {DEVICE}")

## CWA Module — IFT Backward Pass

The Curie-Weiss Activation is defined via a mean-field fixed point: at each forward pass, the network iteratively solves `m* = mean(tanh(x + J·m*))` for up to `K_max` steps. The backward pass uses the Implicit Function Theorem (IFT) to compute exact gradients without differentiating through the iteration loop — making training efficient and stable.

Key parameter: `J` (learned, constrained to (0,1) via sigmoid). The quantity `J·s̄` (where `s̄` is the mean sech² of activations) is the mean-field coupling strength. When `J·s̄ → 1`, the network approaches a phase transition (criticality).

In [ ]:
class CWAFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, J_raw: torch.Tensor, K_max: int = 50):
        J = torch.sigmoid(J_raw)  # scalar in (0,1)
        n = x.shape[-1]

        m = torch.zeros(*x.shape[:-1], 1, device=x.device, dtype=x.dtype)

        for k in range(K_max):
            h     = x + J * m
            m_new = torch.tanh(h).mean(dim=-1, keepdim=True)
            s_bar = (1.0 / torch.cosh(h)).pow(2).mean(dim=-1, keepdim=True)
            j_s_bar = J * s_bar
            delta = (1e-4 * (1.0 - j_s_bar.clamp(max=0.9999))).clamp(min=1e-8)
            if (m_new - m).abs().max() < delta.max():
                m = m_new
                break
            m = m_new

        m_star  = m.detach()
        h_star  = x + J * m_star
        s_k     = (1.0 / torch.cosh(h_star)).pow(2)  # (batch, n)
        s_bar   = s_k.mean(dim=-1, keepdim=True)       # (batch, 1)
        j_s_bar = (J * s_bar).squeeze(-1)              # (batch,)
        y       = torch.tanh(h_star)

        ift_triggered = (j_s_bar >= 0.8).sum().item()

        ctx.save_for_backward(x, J_raw, m_star, s_k, s_bar)
        ctx.K_max = K_max
        ctx.ift_triggered = ift_triggered
        ctx.j_s_bar_mean  = j_s_bar.mean().item()
        return y, j_s_bar.mean().detach(), torch.tensor(float(ift_triggered))

    @staticmethod
    def backward(ctx, grad_y, grad_jsbar, grad_ift):
        x, J_raw, m_star, s_k, s_bar = ctx.saved_tensors
        J       = torch.sigmoid(J_raw)
        n       = x.shape[-1]
        j_s_bar = J * s_bar  # (batch, 1)
        denom   = (1.0 - j_s_bar).clamp(min=1e-6)

        # Σ_gs = Σ_k g_k * s_k  per sample
        sum_gs = (grad_y * s_k).sum(dim=-1, keepdim=True)  # (batch, 1)

        # ∂L/∂x_k = s_k * [g_k + J * sum_gs / (n * denom)]
        grad_x = s_k * (grad_y + J * sum_gs / (n * denom))

        # ∂L/∂J = m* * s̄ * sum_gs / denom  (summed over batch)
        grad_J_sum = (m_star * s_bar * sum_gs / denom).sum()
        # chain rule: J = sigmoid(J_raw)
        grad_J_raw = grad_J_sum * J * (1.0 - J)

        return grad_x, grad_J_raw, None


class CWALayer(nn.Module):
    """Curie-Weiss Activation with IFT backward."""

    def __init__(self, K_max: int = 50):
        super().__init__()
        self.J_raw  = nn.Parameter(torch.zeros(1))
        self.K_max  = K_max
        self.last_j_s_bar       = 0.0
        self.last_ift_triggered = 0
        self.last_iters         = 0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y, j_s_bar, ift_trig = CWAFunction.apply(x, self.J_raw, self.K_max)
        self.last_j_s_bar        = j_s_bar.item()
        self.last_ift_triggered += int(ift_trig.item())
        return y

## CWA-ShiftOnly (Sub-Exp B Ablation)

The shift-only ablation replaces the full CWA fixed-point with a single detached mean-shift: `y_i = tanh(x_i + c)` where `c = J_frozen · mean(tanh(x))`. There is no backprop through `c` — this tests whether the coupling mechanism itself (beyond just the shift) contributes to performance.

In [ ]:
class CWAShiftOnlyLayer(nn.Module):
    """Constant-shift ablation: y_i = tanh(x_i + c), c = J_frozen * mean(tanh(x)). No backprop through c."""

    def __init__(self, J_frozen: float = 0.5):
        super().__init__()
        self.J_frozen = J_frozen

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            c = self.J_frozen * torch.tanh(x).mean(dim=-1, keepdim=True)
        return torch.tanh(x + c.detach())

## MLP Architecture

Unnormalized 10-layer MLP (no BatchNorm, LayerNorm, or Dropout) to isolate the effect of the activation function. The weight init scheme (`kaiming` vs `small` σ=0.01) is a key variable in Sub-Exp A.

In [ ]:
def build_mlp(
    depth: int = 10,
    hidden: int = 256,
    in_dim: int = 3072,
    out_dim: int = 10,
    act: str = "cwa",
    weight_init: str = "kaiming",
    K_max: int = 50,
) -> nn.Sequential:
    """Build unnormalized MLP (no BatchNorm, LayerNorm, Dropout)."""
    def make_act():
        if act == "cwa":         return CWALayer(K_max=K_max)
        elif act == "shift_only": return CWAShiftOnlyLayer(J_frozen=0.5)
        elif act == "tanh":       return nn.Tanh()
        elif act == "gelu":       return nn.GELU()
        else: raise ValueError(f"Unknown act: {act}")

    dims   = [in_dim] + [hidden] * (depth - 1) + [out_dim]
    layers = []
    for i in range(len(dims) - 1):
        linear = nn.Linear(dims[i], dims[i + 1])
        if weight_init == "small":
            nn.init.normal_(linear.weight, mean=0.0, std=0.01)
            nn.init.zeros_(linear.bias)
        else:  # kaiming
            nn.init.kaiming_uniform_(linear.weight, a=math.sqrt(5))
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(linear.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(linear.bias, -bound, bound)
        layers.append(linear)
        if i < len(dims) - 2:
            layers.append(make_act())
    return nn.Sequential(*layers)

## Smoke Test

Validates the CWA mechanism in isolation (no CIFAR-10 download needed): forward pass shape, IFT backward, CWAShiftOnly no-parameter check, MLP layer counts, and small-init weight std. Uses the small config values defined above.

In [ ]:
def smoke_test(device):
    logger.info("--- Smoke Test ---")

    # 1. CWA forward
    layer = CWALayer(K_max=SMOKE_K_MAX).to(device)
    x = torch.randn(SMOKE_BATCH, SMOKE_HIDDEN, device=device)
    y, j_s_bar, ift = CWAFunction.apply(x, layer.J_raw, SMOKE_K_MAX)
    assert y.shape == (SMOKE_BATCH, SMOKE_HIDDEN), f"Expected ({SMOKE_BATCH},{SMOKE_HIDDEN}), got {y.shape}"
    assert 0.0 < j_s_bar.item() < 1.0, f"j_s_bar={j_s_bar.item()} out of range"
    y.sum().backward()
    assert layer.J_raw.grad is not None and layer.J_raw.grad.abs().item() > 0, "J_raw.grad is zero"
    logger.info(f"  CWA forward OK — j_s_bar={j_s_bar.item():.4f}")

    # 2. CWA-ShiftOnly no-param check
    shift_layer = CWAShiftOnlyLayer().to(device)
    x2 = torch.randn(SMOKE_BATCH, SMOKE_HIDDEN, device=device, requires_grad=True)
    y2 = shift_layer(x2)
    assert list(shift_layer.parameters()) == [], "CWAShiftOnly should have no parameters"
    y2.sum().backward()
    assert x2.grad is not None, "Input x2 should still receive gradient"
    logger.info("  CWAShiftOnly no-param OK")

    # 3. MLP build checks (using SMOKE_ config values)
    m_cwa   = build_mlp(depth=SMOKE_DEPTH, hidden=SMOKE_HIDDEN, act="cwa",        K_max=SMOKE_K_MAX)
    m_shift = build_mlp(depth=SMOKE_DEPTH, hidden=SMOKE_HIDDEN, act="shift_only", K_max=SMOKE_K_MAX)
    m_tanh  = build_mlp(depth=SMOKE_DEPTH, hidden=SMOKE_HIDDEN, act="tanh",       K_max=SMOKE_K_MAX)
    n_cwa   = sum(1 for m in m_cwa.modules()   if isinstance(m, CWALayer))
    n_shift = sum(1 for m in m_shift.modules() if isinstance(m, CWAShiftOnlyLayer))
    n_tanh  = sum(1 for m in m_tanh.modules()  if isinstance(m, nn.Tanh))
    expected = SMOKE_DEPTH - 1  # one activation per hidden layer
    assert n_cwa   == expected, f"Expected {expected} CWA layers, got {n_cwa}"
    assert n_shift == expected, f"Expected {expected} ShiftOnly layers, got {n_shift}"
    assert n_tanh  == expected, f"Expected {expected} Tanh layers, got {n_tanh}"
    logger.info(f"  MLP build OK — cwa:{n_cwa} shift:{n_shift} tanh:{n_tanh}")

    # 4. Small-init weight std
    m_small = build_mlp(depth=SMOKE_DEPTH, hidden=SMOKE_HIDDEN, act="cwa", weight_init="small", K_max=SMOKE_K_MAX)
    linears = [m for m in m_small.modules() if isinstance(m, nn.Linear)]
    std_val = linears[0].weight.std().item()
    assert abs(std_val - 0.01) < 0.005, f"small init std={std_val:.4f} expected ~0.01"
    logger.info(f"  Small init std={std_val:.5f} OK")

    logger.info("--- Smoke Test PASSED ---")

smoke_test(DEVICE)

## Statistical Analysis Functions

These functions aggregate per-seed results, compute 95% CIs, and run paired t-tests across conditions. The `determine_mechanistic_verdict` function applies the experimental decision rules for both sub-experiments.

In [ ]:
def aggregate_results(records: list, sub_exp_name: str, metric: str = "final_test_acc") -> dict:
    by_cond = defaultdict(list)
    for r in records:
        if r["sub_exp"] == sub_exp_name:
            by_cond[r["condition"]].append(r[metric])

    out = {}
    for cond, vals in by_cond.items():
        vals_arr = np.array(vals)
        n    = len(vals_arr)
        mean = float(np.mean(vals_arr))
        std  = float(np.std(vals_arr, ddof=1)) if n > 1 else 0.0
        if n > 1:
            se     = std / math.sqrt(n)
            t_crit = stats.t.ppf(0.975, df=n - 1)
            ci     = (mean - t_crit * se, mean + t_crit * se)
        else:
            ci = (mean, mean)
        out[cond] = {"mean": mean, "std": std, "ci_95": list(ci), "n": n, "values": list(vals_arr)}
    return out


def paired_ttest(records: list, sub_exp: str, cond_a: str, cond_b: str, metric: str = "final_test_acc") -> dict:
    a = [r[metric] for r in records if r["sub_exp"] == sub_exp and r["condition"] == cond_a]
    b = [r[metric] for r in records if r["sub_exp"] == sub_exp and r["condition"] == cond_b]
    if len(a) == len(b) and len(a) >= 2:
        t_stat, p_val = stats.ttest_rel(a, b)
        return {"t_stat": float(t_stat), "p_val": float(p_val), "n_pairs": len(a)}
    return {"t_stat": None, "p_val": None, "n_pairs": len(a)}


def determine_mechanistic_verdict(agg_B: dict, agg_A: dict) -> dict:
    cwa_full_acc   = agg_B.get("cwa_full",       {}).get("mean", 0.0)
    shift_only_acc = agg_B.get("cwa_shift_only", {}).get("mean", 0.0)
    tanh_acc       = agg_B.get("pure_tanh",       {}).get("mean", 0.0)

    THRESH = 0.005
    if abs(shift_only_acc - cwa_full_acc) <= THRESH:
        verdict_B = "bias_dominant"
    elif shift_only_acc > cwa_full_acc + THRESH:
        verdict_B = "coupling_harmful"
    elif abs(shift_only_acc - tanh_acc) <= THRESH:
        verdict_B = "capacity_only"
    else:
        verdict_B = "ambiguous"

    descriptions = {
        "bias_dominant":     "Coupling loss is entirely from mean shift; fixed-point adds no value.",
        "coupling_harmful":  "Iterative feedback actively hurts; one-shot shift is better.",
        "capacity_only":     "Shift has no effect; accuracy gap is pure capacity/optimization.",
        "ambiguous":         "No clear verdict; intermediate regime.",
    }

    # Sub-Exp A: check small-init criticality
    cwa_small_jsbar = agg_A.get("cwa_small_init", {}).get("mean", 0.0)
    if cwa_small_jsbar > 0.7:
        verdict_A = "init_unlocks_criticality"
    else:
        verdict_A = "init_does_not_help"

    return {
        "sub_exp_B_verdict":     verdict_B,
        "sub_exp_B_description": descriptions.get(verdict_B, ""),
        "sub_exp_A_verdict":     verdict_A,
    }

## Load and Reconstruct Results from Precomputed Data

The mini dataset stores epoch-level accuracy and J·s̄ values per seed. This cell reconstructs the per-(seed, condition) records needed by the stat functions, and extracts trajectories for plotting.

In [ ]:
# Separate examples by sub-experiment
examples_A = [e for e in examples if e.get("metadata_sub_exp") == "A_small_weight_init"]
examples_B = [e for e in examples if e.get("metadata_sub_exp") == "B_shift_ablation"]

# Reconstruct per-(seed, condition) records from epoch-level examples
def reconstruct_records_A(exs):
    """Group Sub-Exp A epoch rows into per-(seed, condition) records."""
    from collections import defaultdict
    traj = defaultdict(lambda: defaultdict(list))   # traj[seed][cond] = [acc_ep1, ...]
    jsbar = defaultdict(lambda: defaultdict(list))  # jsbar[seed][cond] = [jsbar_ep1, ...]
    for e in sorted(exs, key=lambda x: (x["metadata_seed"], int(x["metadata_epoch"]))):
        seed = e["metadata_seed"]
        for cond, key in [("cwa_small_init",   "predict_cwa_small_init"),
                           ("gelu_small_init",  "predict_gelu_small_init"),
                           ("cwa_kaiming_init", "predict_cwa_kaiming_init")]:
            if key in e:
                traj[seed][cond].append(float(e[key]))
        for cond, key in [("cwa_small_init",   "metadata_j_s_bar_cwa_small_init"),
                           ("cwa_kaiming_init", "metadata_j_s_bar_cwa_kaiming_init")]:
            if key in e:
                jsbar[seed][cond].append(float(e[key]))

    records = []
    for seed, conds in traj.items():
        for cond, accs in conds.items():
            jsbars = jsbar[seed].get(cond, [0.0] * len(accs))
            records.append({
                "sub_exp": "A_small_weight_init",
                "condition": cond,
                "seed": seed,
                "final_test_acc": accs[-1] if accs else 0.0,
                "test_acc_trajectory": accs,
                "j_s_bar_trajectory": jsbars,
                "max_j_s_bar_achieved": max(jsbars) if jsbars else 0.0,
            })
    return records


def reconstruct_records_B(exs):
    """Group Sub-Exp B epoch rows into per-(seed, condition) records."""
    from collections import defaultdict
    traj  = defaultdict(lambda: defaultdict(list))
    jsbar = defaultdict(lambda: defaultdict(list))
    for e in sorted(exs, key=lambda x: (x["metadata_seed"], int(x["metadata_epoch"]))):
        seed = e["metadata_seed"]
        for cond, key in [("cwa_full",       "predict_cwa_full"),
                           ("cwa_shift_only", "predict_cwa_shift_only"),
                           ("pure_tanh",      "predict_pure_tanh")]:
            if key in e:
                traj[seed][cond].append(float(e[key]))
        for cond, key in [("cwa_full", "metadata_j_s_bar_cwa_full")]:
            if key in e:
                jsbar[seed][cond].append(float(e[key]))

    records = []
    for seed, conds in traj.items():
        for cond, accs in conds.items():
            jsbars = jsbar[seed].get(cond, [0.0] * len(accs))
            final_gr = accs[-1]  # proxy
            records.append({
                "sub_exp": "B_shift_ablation",
                "condition": cond,
                "seed": seed,
                "final_test_acc": accs[-1] if accs else 0.0,
                "test_acc_trajectory": accs,
                "j_s_bar_trajectory": jsbars,
                "grad_ratio_abs_deviation": 0.0,  # not stored in epoch-level mini
            })
    return records


records_A = reconstruct_records_A(examples_A)
records_B = reconstruct_records_B(examples_B)
all_records = records_A + records_B

print(f"Reconstructed {len(records_A)} Sub-Exp A records, {len(records_B)} Sub-Exp B records")

# Run statistical analysis
agg_B       = aggregate_results(all_records, "B_shift_ablation",    "final_test_acc")
agg_A_acc   = aggregate_results(all_records, "A_small_weight_init", "final_test_acc")
agg_A_jsbar = aggregate_results(all_records, "A_small_weight_init", "max_j_s_bar_achieved")

ttest_full_vs_shift = paired_ttest(all_records, "B_shift_ablation", "cwa_full",       "cwa_shift_only")
ttest_shift_vs_tanh = paired_ttest(all_records, "B_shift_ablation", "cwa_shift_only", "pure_tanh")
ttest_full_vs_tanh  = paired_ttest(all_records, "B_shift_ablation", "cwa_full",       "pure_tanh")

verdict = determine_mechanistic_verdict(agg_B, agg_A_jsbar)

print("\n=== Sub-Exp A: Small-Weight Init ===")
print(f"  Verdict: {verdict['sub_exp_A_verdict']}")
for cond, v in agg_A_acc.items():
    jsbar_v = agg_A_jsbar.get(cond, {})
    print(f"  {cond:20s}  acc={v['mean']:.4f}±{v['std']:.4f}  max_j_s_bar={jsbar_v.get('mean',0):.4f}")

print("\n=== Sub-Exp B: Shift Ablation ===")
print(f"  Verdict: {verdict['sub_exp_B_verdict']} — {verdict['sub_exp_B_description']}")
for cond, v in agg_B.items():
    print(f"  {cond:20s}  acc={v['mean']:.4f}±{v['std']:.4f}  CI=[{v['ci_95'][0]:.4f},{v['ci_95'][1]:.4f}]")

print(f"\n  t-test CWA-Full vs Shift-Only: t={ttest_full_vs_shift['t_stat']:.3f}  p={ttest_full_vs_shift['p_val']:.3f}")
print(f"  t-test Shift-Only vs Tanh:      t={ttest_shift_vs_tanh['t_stat']:.3f}  p={ttest_shift_vs_tanh['p_val']:.3f}")
print(f"  t-test CWA-Full vs Tanh:        t={ttest_full_vs_tanh['t_stat']:.3f}  p={ttest_full_vs_tanh['p_val']:.3f}")

## Visualization

Four plots summarizing the mechanistic findings:
1. **Sub-Exp A accuracy trajectories** — seed=42: CWA-small-init vs CWA-Kaiming vs GELU-small-init
2. **J·s̄ trajectories** — tracks how close the mean-field coupling gets to criticality (target >0.7)
3. **Sub-Exp B accuracy trajectories** — seed=42: CWA-Full vs CWA-ShiftOnly vs Pure-Tanh
4. **Final accuracy bar chart** — mean ± std across all 3 seeds for each condition

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("CWA Mechanistic Sub-Experiments on CIFAR-10", fontsize=14, fontweight="bold")

# ── Panel 1: Sub-Exp A — accuracy trajectories (seed=42) ──────────────────────
ax1 = axes[0, 0]
colors_A = {"cwa_small_init": "#e74c3c", "gelu_small_init": "#2ecc71", "cwa_kaiming_init": "#3498db"}
labels_A = {"cwa_small_init": "CWA (σ=0.01)", "gelu_small_init": "GELU (σ=0.01)", "cwa_kaiming_init": "CWA (Kaiming)"}
for r in records_A:
    if r["seed"] == "42":
        traj = r["test_acc_trajectory"]
        ax1.plot(range(1, len(traj)+1), traj,
                 color=colors_A.get(r["condition"], "gray"),
                 label=labels_A.get(r["condition"], r["condition"]),
                 linewidth=2)
ax1.set_title("Sub-Exp A: Init Scheme (seed=42)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Test Accuracy")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0.1, 0.6)

# ── Panel 2: Sub-Exp A — J·s̄ trajectories (seed=42) ─────────────────────────
ax2 = axes[0, 1]
colors_jsbar = {"cwa_small_init": "#e74c3c", "cwa_kaiming_init": "#3498db"}
for r in records_A:
    if r["seed"] == "42" and r["condition"] in colors_jsbar:
        traj = r["j_s_bar_trajectory"]
        if traj and any(v > 0 for v in traj):
            ax2.plot(range(1, len(traj)+1), traj,
                     color=colors_jsbar[r["condition"]],
                     label=labels_A[r["condition"]],
                     linewidth=2)
ax2.axhline(y=0.7, color="black", linestyle="--", linewidth=1.5, label="Criticality threshold (0.7)")
ax2.axhline(y=1.0, color="gray",  linestyle=":",  linewidth=1.0, label="Phase transition (1.0)")
ax2.set_title("Sub-Exp A: J·s̄ Coupling Strength (seed=42)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("J·s̄")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0.0, 1.1)

# ── Panel 3: Sub-Exp B — accuracy trajectories (seed=42) ──────────────────────
ax3 = axes[1, 0]
colors_B = {"cwa_full": "#9b59b6", "cwa_shift_only": "#e67e22", "pure_tanh": "#1abc9c"}
labels_B = {"cwa_full": "CWA-Full", "cwa_shift_only": "CWA-ShiftOnly", "pure_tanh": "Pure-Tanh"}
for r in records_B:
    if r["seed"] == "42":
        traj = r["test_acc_trajectory"]
        ax3.plot(range(1, len(traj)+1), traj,
                 color=colors_B.get(r["condition"], "gray"),
                 label=labels_B.get(r["condition"], r["condition"]),
                 linewidth=2)
ax3.set_title("Sub-Exp B: Shift Ablation (seed=42)")
ax3.set_xlabel("Epoch")
ax3.set_ylabel("Test Accuracy")
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.set_ylim(0.3, 0.55)

# ── Panel 4: Final accuracy bar chart (all seeds, mean ± std) ─────────────────
ax4 = axes[1, 1]
all_conds = []
all_means = []
all_stds  = []
bar_colors = []

for cond, v in agg_A_acc.items():
    all_conds.append(labels_A.get(cond, cond).replace(" (", "\n("))
    all_means.append(v["mean"])
    all_stds.append(v["std"])
    bar_colors.append(colors_A.get(cond, "gray"))

for cond, v in agg_B.items():
    all_conds.append(labels_B.get(cond, cond))
    all_means.append(v["mean"])
    all_stds.append(v["std"])
    bar_colors.append(colors_B.get(cond, "gray"))

x = range(len(all_conds))
bars = ax4.bar(x, all_means, yerr=all_stds, color=bar_colors, capsize=4, alpha=0.85)
ax4.set_xticks(list(x))
ax4.set_xticklabels(all_conds, fontsize=7, rotation=15, ha="right")
ax4.set_ylabel("Test Accuracy")
ax4.set_title("Final Test Accuracy (mean ± std, 3 seeds)")
ax4.grid(True, alpha=0.3, axis="y")
ax4.set_ylim(0.35, 0.58)

# Annotate verdict
ax4.text(0.02, 0.97,
         f"Sub-Exp A: {verdict['sub_exp_A_verdict']}\nSub-Exp B: {verdict['sub_exp_B_verdict']}",
         transform=ax4.transAxes, fontsize=8, va="top",
         bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.tight_layout()
plt.savefig("cwa_mechanistic_results.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: cwa_mechanistic_results.png")